### Exercise 1: Gemini manual tool calling

In [1]:
import datetime
import json
import os
from typing import Callable

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
print("Gemini API key loaded.")

Gemini API key loaded.


In [2]:
def current_date_tool() -> str:
    return datetime.date.today().isoformat()


def weather_forecast_tool(country: str, city: str, date: str) -> str:
    if country.lower() in {"united kingdom", "uk", "england"}:
        return "Fog and rain"
    return "Sunshine"

In [3]:
def get_tool_definitions() -> tuple[list[dict], dict[str, Callable]]:
    tool_definitions = [
        {
            "type": "function",
            "function": {
                "name": "get_current_date",
                "description": 'Get current date in the format "Year-Month-Day" (YYYY-MM-DD).',
                "parameters": {},
            },
        },
        {
            "type": "function",
            "function": {
                "name": "get_weather_forecast",
                "description": "Get weather forecast at given country, city, and date.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "country": {
                            "type": "string",
                            "description": "The country the city is in.",
                        },
                        "city": {
                            "type": "string",
                            "description": "The city to get the weather for.",
                        },
                        "date": {
                            "type": "string",
                            "description": (
                                "The date to get the weather for, "
                                'in the format "Year-Month-Day" (YYYY-MM-DD). '
                                "At most 4 weeks into the future."
                            ),
                        },
                    },
                    "required": ["country", "city", "date"],
                },
            },
        },
    ]

    tool_name_to_callable = {
        "get_current_date": current_date_tool,
        "get_weather_forecast": weather_forecast_tool,
    }
    return tool_definitions, tool_name_to_callable

Gemini request loop

In [4]:
def make_llm_request(prompt: str) -> str:
    client = OpenAI(
        api_key=os.environ["GEMINI_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        timeout=60.0,
        max_retries=1,
    )

    messages = [
        {"role": "system", "content": "You are a weather assistant."},
        {"role": "user", "content": prompt},
    ]
    tool_definitions, tool_name_to_func = get_tool_definitions()

    print(f"PROMPT: {prompt}\n")
    for step in range(1, 11):
        response = client.chat.completions.create(
            model="gemini-3.5-flash",
            messages=messages,
            tools=tool_definitions,
            tool_choice="auto",
            max_completion_tokens=1000,
            reasoning_effort="low",
        )
        resp_message = response.choices[0].message
        message_data = resp_message.model_dump(exclude_none=True)
        messages.append(message_data)

        print(f"GENERATED MESSAGE {step}:")
        print(json.dumps(message_data, indent=2))
        print()

        if not resp_message.tool_calls:
            return resp_message.content or ""

        for tool_call in resp_message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            func_result = tool_name_to_func[func_name](**func_args)

            print("TOOL RESULT:")
            print(
                json.dumps(
                    {
                        "name": func_name,
                        "arguments": func_args,
                        "result": func_result,
                    },
                    indent=2,
                )
            )
            print()

            messages.append(
                {
                    "role": "tool",
                    "content": json.dumps(func_result),
                    "tool_call_id": tool_call.id,
                }
            )

    return f"Could not resolve request, last response: {resp_message.content}"

### Birmingham test

In [5]:
birmingham_answer = make_llm_request(
    "What will be weather in Birmingham, UK in two weeks?"
)
print("FINAL RESPONSE:")
print(birmingham_answer)

PROMPT: What will be weather in Birmingham, UK in two weeks?

GENERATED MESSAGE 1:
{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "3k2fn3y8",
      "function": {
        "arguments": "{}",
        "name": "get_current_date"
      },
      "type": "function",
      "extra_content": {
        "google": {
          "thought_signature": "EuQCCuECAQw51sdtIGMqPGyCWo9UTbis2agqELWhUd8dvShgJrwXbwiznq8XJBvZqLngI2NFJGDjnm/SU3Mrv5WrW6A+G3II3zVq029d8+NvF+4S4Qh8WkKBqXp8MkuCBDmUW3Jhs2ahERR80J0NgCF4CCg4hIpVih4wVjVSoqfUVxWmo5usSeCO/o9XOYOLmmM+Ehj2mTmePggFrshKQAjkBTjNOjD215pi2p4eyRBFC68WZ7NKlvIBP8rKukvibsqMEnULX8pIm4a/E/7VoBjVAKbC1nfSH5R3J0qqfJJ3YQlodGa7ChqSdCvENt0wK9NFkpU0watEG8OLf8B+YirKrOdhBINYHZYOUu80I5ULWefvy12+aRxXMG61TKevtRM5oJeOqa9soxdi9vnLdrzKkmP3vWubpJUwjdgjTUpzwZJfurnbvvw2nWTtx6kfOALtmxWtejSb/tU7CQZjxIMbyzM93h4="
        }
      }
    }
  ]
}

TOOL RESULT:
{
  "name": "get_current_date",
  "arguments": {},
  "result": "2026-06-12"
}

GENERATED MESSAGE 2:
{
  "role": "assistant",

### Warsaw test

In [6]:
warsaw_answer = make_llm_request(
    "What will be weather in Warsaw the day after tomorrow?"
)
print("FINAL RESPONSE:")
print(warsaw_answer)

PROMPT: What will be weather in Warsaw the day after tomorrow?

GENERATED MESSAGE 1:
{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "sg5xt7d8",
      "function": {
        "arguments": "{}",
        "name": "get_current_date"
      },
      "type": "function",
      "extra_content": {
        "google": {
          "thought_signature": "EqkCCqYCAQw51sd5bllmNswKcOXa7nde+wrvQYlLBKAs8i8EVXOi7yeNmuTq6spmu2/r8hRW7FbFvnxwOtMDoryP07ZHpPzlXOcMmwcuxlkoyLQ6elMpvJwOmcugELIyreTN20RZ/3sQI/AlgKfxEs8ztVcgmchMl3r78wwmMHzgkUJCZx7yX9lO5pN8VGzzT2fJDH27+9svEjI0bpJRnqO7N8bQ0O8q+w0hOJ3d6Q8LObJZwSl+Ylat+1w9PA5sceKVjUl6RzajIt/N6QRtrqsVFCAQdnCJjvgnAjgp58IXNfIs7JL8+Trfip2JN4jmmeAlgUc57i2dwf+YT+cmmjd7D/sJp3t5p87EPJi9hbUkdqLR3JxO776ePizpAMO+wbB3O+yMw9ssp9N4"
        }
      }
    }
  ]
}

TOOL RESULT:
{
  "name": "get_current_date",
  "arguments": {},
  "result": "2026-06-12"
}

GENERATED MESSAGE 2:
{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "ybrb9frh",
      "function": {
        "

### New York test

In [7]:
new_york_answer = make_llm_request(
    "What will be weather in New York in two months?"
)
print("FINAL RESPONSE:")
print(new_york_answer)

PROMPT: What will be weather in New York in two months?

GENERATED MESSAGE 1:
{
  "role": "assistant",
  "tool_calls": [
    {
      "id": "l1yc8rtg",
      "function": {
        "arguments": "{}",
        "name": "get_current_date"
      },
      "type": "function",
      "extra_content": {
        "google": {
          "thought_signature": "EvUCCvICAQw51sdcYEzw+p6XUS+Z/nJtKnjmbkWTbl1yMLl5Dik1b6ocPiNetdz1zueBy5OXHtJlixdtMCL2JBi0AlE+Czldmx62neHjKo1UD7005lIBTVcPCcwiN55j1Nd7t5A4re6O2IU9VOpTKE3T+sONSj/3w891/XJtwgO2VyGVka2M/tdBW/hEL19Y75pMxkrrgCvnUBVzPl7JxLlFWeLv5ZFxNYVCKkLRGyJH9bQ1OOxO0OYwuCVlE7sAiPfa8nEtRbsu9H0zi2W+H2FWLfgx79/LOzrvv85OsS5fwQ+ox8ia5qfjLV29S+eTnVdLd0cDaNyyJaSTFZtC9JmE9+bzRYDjlkyH9iR8MjikufNZshHwSQ+say7drqHSDaS7jIpWKf0GK6PPjL+lb0HPVjSq+m7q/ntmEIg09CymMgWipbQ2mmau1bzJq24DPcEP2aXW0zcCw9UUaFC3/Yzsa3lFrj6Z0CRm5ah/+DhM3mxjAv324w=="
        }
      }
    }
  ]
}

TOOL RESULT:
{
  "name": "get_current_date",
  "arguments": {},
  "result": "2026-06-12"
}

GENERATED MESSAGE 2:
{
  "

### Warsaw

1. The model first called **`get_current_date`** with no arguments.
2. The tool returned the current date **2026-06-12**.
3. The model calculated the day after tomorrow as **2026-06-14** and used it as the `date` argument when calling **`get_weather_forecast`** with country **Poland** and city **Warsaw**.
4. The weather tool returned **`Sunshine`**.
5. Gemini used that tool result to produce the final answer: **"The weather in Warsaw the day after tomorrow (June 14, 2026) is expected to be sunny."**


### New York in two months

The request is outside the tool's four-week range. Gemini first called **`get_current_date`**, which returned **2026-06-12**. It calculated that two months later would be approximately **2026-08-12**, but it did **not** call `get_weather_forecast`. Instead, it explained that forecasts are available only up to four weeks in advance and asked the user to try again closer to the date. This shows that the model correctly respected the constraint declared in the tool description.

Final response: **"I can only provide weather forecasts for up to 4 weeks into the future. Since you are asking for the weather in New York in two months (around August 12, 2026), I am unable to retrieve that forecast. Please ask closer to the date!"**